# ReAct Research Copilot — Experimentation Notebook

This notebook walks through the full pipeline:
1. Corpus Ingestion
2. Chunking & Embedding
3. Vector Search
4. ReAct Agent
5. Evaluation Results

In [1]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv

# Set project root absolutely
PROJECT_ROOT = Path("..").resolve()
os.chdir(PROJECT_ROOT)
load_dotenv(PROJECT_ROOT / ".env")
sys.path.insert(0, str(PROJECT_ROOT))

from copilot.config import settings
from copilot.ingestion.ingest_pipeline import load_corpus
from copilot.retrieval.chunking import chunk_documents
from copilot.retrieval.vector_store import load_index, search
from copilot.agent.tools import search_corpus, open_file
from copilot.agent.react_agent import run_react_agent

print("All imports successful")
print(f"Project root: {PROJECT_ROOT}")
print(f"LLM Model: {settings.llm_model}")
print(f"Embed Model: {settings.embed_model}")

All imports successful
Project root: D:\ai-projects\bia-capstone-projects\react-research-copilot
LLM Model: meta/llama-3.1-8b-instruct
Embed Model: nvidia/llama-nemotron-embed-1b-v2


## 1. Corpus Ingestion
Load all 39 documents from the corpus folder.

In [2]:
docs = load_corpus()
print(f"Total documents: {len(docs)}")
print(f"\nSample document:")
print(f"  File: {docs[0]['file']}")
print(f"  Type: {docs[0]['type']}")
print(f"  Text preview: {docs[0]['text'][:200]}")

Loading corpus: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 39/39 [00:00<00:00, 1988.35it/s]

Total documents: 39

Sample document:
  File: 01_ai_evals.md
  Type: markdown
  Text preview: # Practical LLM Evaluation Basics (Sample Corpus)

**Document ID:** 01_ai_evals.md

## Reference Sentences
Evaluation should separate retrieval quality from generation quality.
Common dimensions inclu


## 2. Chunking
Split documents into smaller chunks for better retrieval precision.

In [3]:
chunks = chunk_documents(docs)
print(f"Total chunks: {len(chunks)}")
print(f"Avg words per chunk: {sum(c['word_count'] for c in chunks) // len(chunks)}")
print(f"\nSample chunk:")
print(f"  ID: {chunks[0]['chunk_id']}")
print(f"  Words: {chunks[0]['word_count']}")
print(f"  Text: {chunks[0]['text'][:200]}")

Total chunks: 103
Avg words per chunk: 38

Sample chunk:
  ID: 01_ai_evals.md::chunk_0
  Words: 43
  Text: # Practical LLM Evaluation Basics (Sample Corpus) **Document ID:** 01_ai_evals.md ## Reference Sentences Evaluation should separate retrieval quality from generation quality. Common dimensions include


## 3. Vector Search
Load the pre-built FAISS index and run a sample search.

In [4]:
index, stored_chunks = load_index()
print(f"Index loaded: {index.ntotal} vectors")

query = "What is faithfulness in LLM evaluation?"
results = search(query, index, stored_chunks)

print(f"\nQuery: {query}")
print(f"Top {len(results)} results:\n")
for i, r in enumerate(results, 1):
    print(f"  {i}. [{r['file']}] score={r['score']:.4f}")
    print(f"     {r['text'][:150]}\n")

Index loaded: 103 vectors

Query: What is faithfulness in LLM evaluation?
Top 5 results:

  1. [04_ai_evals.md] score=1.0435
     # Practical LLM Evaluation Basics (Sample Corpus) **Document ID:** 04_ai_evals.md **Purpose:** Synthetic, test-only document for a research copilot ca

  2. [22_ai_evals.md] score=1.0551
     # Practical LLM Evaluation Basics (Sample Corpus) **Document ID:** 22_ai_evals.md **Purpose:** Synthetic, test-only document for a research copilot ca

  3. [01_ai_evals.md] score=1.0562
     # Practical LLM Evaluation Basics (Sample Corpus) **Document ID:** 01_ai_evals.md ## Reference Sentences Evaluation should separate retrieval quality 

  4. [13_ai_evals.md] score=1.0626
     # Practical LLM Evaluation Basics (Sample Corpus) **Document ID:** 13_ai_evals.md **Purpose:** Synthetic, test-only document for a research copilot ca

  5. [10_ai_evals.md] score=1.0641
     # Practical LLM Evaluation Basics (Sample Corpus) **Document ID:** 10_ai_evals.md **Purpose:** Synthet

## 4. ReAct Agent
Run the full ReAct loop on a sample question.

In [5]:
result = run_react_agent("What are the three common evaluation dimensions for LLM outputs?")

print("\nAnswer:")
print(result["answer"])
print(f"\nSteps used: {result['step_count']}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Question: What are the three common evaluation dimensions for LLM outputs?                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Step 1

Step 1 took 0.43s

Action: search_corpus({'query': 'evaluation dimensions LLM outputs'})

Step 1 took 0.89s

Observe: [04_ai_evals.md]
# Practical LLM Evaluation Basics (Sample Corpus) **Document ID:** 04_ai_evals.md **Purpose:** Synthetic, test-only
document for a research copilot capstone. ## Summary Evaluation should separate retrieval quality from generation 
quality. Common dimensions include relevance, faithfu...

Step 2

Step 2 took 0.55s

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Final Answer:                                                                                                   │
│ The three common evaluation dimensions for LLM outputs are:                                                     │
│                                                                                                                 │
│ 1. Relevance                                                                                                    │
│ 2. Faithfulness (groundedness)                                                                                  │
│ 3. Robustness                                                                                                   │
│                                                                                                                 │
│ [04_ai_evals.md, 22_ai_evals.md, 10_ai_evals.md, 13_ai_evals.md, 01_ai_evals.md]                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Answer:
The three common evaluation dimensions for LLM outputs are:

1. Relevance
2. Faithfulness (groundedness)
3. Robustness

[04_ai_evals.md, 22_ai_evals.md, 10_ai_evals.md, 13_ai_evals.md, 01_ai_evals.md]

Steps used: 1


## 5. Evaluation Results
Compare baseline vs ReAct grounded precision scores.

In [17]:
import pandas as pd
import glob

runs_dir = "outputs/runs"
files = glob.glob(f"{runs_dir}/*.csv")

baseline_files = sorted([f for f in files if "baseline" in f])
react_files = sorted([f for f in files if "react" in f])

print("Baseline files:", baseline_files)
print("React files:", react_files)

baseline_df = pd.read_csv(baseline_files[-1])
react_df = pd.read_csv(react_files[-1])

print("\nEVALUATION SUMMARY")
print("=" * 40)
print(f"Baseline grounded precision: {baseline_df['grounded_precision'].mean():.2f}")
print(f"ReAct    grounded precision: {react_df['grounded_precision'].mean():.2f}")
print(f"Improvement: +{react_df['grounded_precision'].mean() - baseline_df['grounded_precision'].mean():.2f}")

print("\nPer-question comparison:")
comparison = pd.DataFrame({
    "question_id": baseline_df["question_id"],
    "baseline_gp": baseline_df["grounded_precision"],
    "react_gp":    react_df["grounded_precision"]
})
print(comparison.to_string(index=False))

Baseline files: ['outputs/runs\\eval_baseline_20260325_164023.csv', 'outputs/runs\\eval_baseline_20260326_074607.csv']
React files: ['outputs/runs\\eval_react_20260325_181118.csv', 'outputs/runs\\eval_react_20260326_074815.csv']

EVALUATION SUMMARY
Baseline grounded precision: 0.37
ReAct    grounded precision: 0.63
Improvement: +0.26

Per-question comparison:
question_id  baseline_gp  react_gp
        Q01         0.22      0.22
        Q02         0.33      0.00
        Q03         0.78      0.78
        Q04         0.44      0.56
        Q05         0.00      0.00
        Q06         0.25      0.25
        Q07         0.60      1.00
        Q08         0.23      0.15
        Q09         0.38      0.50
        Q10         0.46      0.92
        Q11         0.36      1.00
        Q12         0.17      0.67
        Q13         0.20      0.10
        Q14         0.60      0.80
        Q15         0.57      1.00
        Q16         0.62      0.88
        Q17         0.30      1.00
        

## 6. Failure Analysis
Questions where ReAct performed poorly.

In [21]:
print("Low precision questions (react_gp < 0.3):\n")
failures = comparison[comparison["react_gp"] < 0.3]
for _, row in failures.iterrows():
    question = baseline_df[baseline_df["question_id"] == row["question_id"]]["question"].values[0]
    print(f"  {row['question_id']}: {question[:60]}")
    print(f"  Baseline GP: {row['baseline_gp']}")
    print(f"  ReAct GP:    {row['react_gp']}")
    print()

Low precision questions (react_gp < 0.3):

  Q01: What does 'faithfulness' mean in LLM evaluation?
  Baseline GP: 0.22
  ReAct GP:    0.22

  Q02: Name two defenses against prompt injection.
  Baseline GP: 0.33
  ReAct GP:    0.0

  Q05: What is the purpose of citations in research assistants?
  Baseline GP: 0.0
  ReAct GP:    0.0

  Q06: In a RAG pipeline, what does reranking do?
  Baseline GP: 0.25
  ReAct GP:    0.25

  Q08: Give two examples of variance drivers.
  Baseline GP: 0.23
  ReAct GP:    0.15

  Q13: List two stages in a RAG pipeline.
  Baseline GP: 0.2
  ReAct GP:    0.1

